## Adding another strategy from the library

In [26]:
import pathlib
import re

import axelrod as axl
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import dask.dataframe as dd
import sklearn
from sklearn.feature_selection import RFE, SequentialFeatureSelector, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import json

In [27]:
with open("./data/original_tournament/fortran_characteristics.json", "r") as f:
    characteristics = json.load(f)

In [28]:
number_of_translated_strategies = sum(characteristic['axelrod-python_class'] is not None for characteristic in characteristics.values())
with open("../paper/assets/number_of_translated_strategies.tex", "w") as f:
    f.write(str(number_of_translated_strategies))

In [29]:
second_tournament_strategies = [
    name for name in characteristics.keys()
    if characteristics[name]["original_rank"] is not None
]

with open("../paper/assets/list_of_original_tournament_players.tex", "w") as f:
    for name in second_tournament_strategies:
        dictionary = characteristics[name]
        author=dictionary["author"]
        original_rank=dictionary["original_rank"]
        f.write(f"\\item {name} - Original rank: {original_rank}. Authored by {author}\n")

In [30]:
def get_turns(filename):
    """
    Read the number of turns if included in the file name
    """
    match = re.search("[0-9]+(?=(_turns))", str(filename))
    return int(match.group(0))

def get_repetitions(filename):
    """
    Read the number of repetitions if included in the file name
    """
    match = re.search("[0-9]+(?=(_repetitions))", str(filename))
    return int(match.group(0))

def read_tournament_repetitions(files, player_names=None):
    """
    Read the scores from a collection of gz files 
    representing repetitions of tournaments.
    """
    number_of_opponents = len(player_names) - 1
    dfs = []
    for gz_path in files:
        dfs.append(pd.read_csv(str(gz_path), header=None).iloc[:,0:number_of_opponents + 1])
        
        turns = get_turns(gz_path)
        
        dfs[-1] /= turns * (number_of_opponents)  # Scale all metrics
        dfs[-1].columns = player_names
        
    df = pd.concat(dfs, ignore_index=True)
    return df

def read_payoff_matrix(files):
    arrays = []
    turns = []
    repetitions = 0
    for gz_path in files:
        repetitions += get_repetitions(gz_path)
        arrays.append(np.array(pd.read_csv(str(gz_path), header=None)))  # Read through pd to deal with float conversion
        turns.append(get_turns(str(gz_path)))
    payoff_matrix = sum(array * turn for turn, array in zip(turns, arrays)) / sum(turns)

    return payoff_matrix, repetitions

def get_indices_of_players(player_names, player_index):
    """
    Returns the indices of the players in `player_names` from `player_index`
    """
    indices = []
    for player in player_names:
        indices.append(list(player_index).index(player))
    return indices

def get_results_of_sub_tournament(
    player_names, 
    payoff_matrix, 
    player_index, 
    characteristics=characteristics,
):
    player_indices = get_indices_of_players(player_names, player_index)
    player_index_mesh = np.ix_(player_indices, player_indices)
    payoff_sub_matrix = payoff_matrix[player_index_mesh]
    mean_payoffs = np.mean(payoff_sub_matrix, axis=1)
    median_payoffs = np.median(payoff_sub_matrix, axis=1)
    df = pd.DataFrame(
        {
            "Name": player_names,
            "Mean payoff": mean_payoffs,
            "Median payoff": median_payoffs,
        }
    )
    df["Rank"] = df["Mean payoff"].rank(ascending=False)
    original_ranks = []
    for name in df["Name"]:
        try:
            original_rank = characteristics[name]['original_rank']
        except KeyError:
            original_rank = None
        original_ranks.append(original_rank)
    df["Original Rank"] = original_ranks
    return df.sort_values("Rank")

def add_superscript_to_name(string):
    if ("Evolved" in string) or ("PSO" in string):
        return string + r"\textsuperscript{\textdagger}"
    return string

In [31]:
original_tournament_data_path = pathlib.Path("./data/original_tournament/")
original_tournament_scores = read_tournament_repetitions(
                                   files=original_tournament_data_path.glob("*scores.gz"), 
                                   player_names=second_tournament_strategies)

In [32]:
full_tournament_data_path = pathlib.Path("./data/full_tournament/")
full_tournament_index = pd.read_csv(
    "./data/full_tournament/players.index",
    names=("Name",),
)
full_tournament_scores = read_tournament_repetitions(
                                   files=full_tournament_data_path.glob("*scores.gz"), 
                                   player_names=full_tournament_index["Name"],
)

In [33]:
full_tournament_player_index = pd.read_csv(
    f"{full_tournament_data_path}/players.index",
    names=("Name",),
)

fortran_player_index = pd.read_csv(
    f"{original_tournament_data_path}/players.index",
    names=("Name",),
)
full_tournament_payoff_matrix, repetitions = read_payoff_matrix(
    full_tournament_data_path.glob("*payoff_matrix.gz")
)

In [34]:
indices = get_indices_of_players(
    player_names=fortran_player_index["Name"], 
    player_index=full_tournament_player_index["Name"],
)

In [35]:
ddf = dd.read_csv("./data/extra_player/main.csv")
ddf.head()

,Name,Mean payoff,Median payoff,Rank,Original Rank,number of new strategies,tournament id,Winner
0,ALLCorALLD,2.119473,2.227812,58.0,NaN,1,1,k92r
1,AON2,2.634555,3.000000,49.0,NaN,1,2,k92r
2,Adaptive Pavlov 2006,2.745572,3.000000,17.0,NaN,1,3,k92r
3,Adaptive Pavlov 2011,2.722915,3.000000,23.0,NaN,1,4,k92r
4,Adaptive,1.932526,1.854349,61.0,NaN,1,5,k92r


In [36]:
ddf.tail()

,Name,Mean payoff,Median payoff,Rank,Original Rank,number of new strategies,tournament id,Winner
950600,Average Copier,2.539951,2.710133,54.0,NaN,3,324252,k92r
950601,"Bush Mosteller: 0.5, 0.5, 3.0, 0.5",1.700886,1.623992,65.0,NaN,3,324252,k92r
950602,PSO Gambler 1_1_1,2.667447,3.000000,36.0,NaN,3,324253,k92r
950603,Average Copier,2.544815,2.727711,53.0,NaN,3,324253,k92r
950604,"Bush Mosteller: 0.5, 0.5, 3.0, 0.5",1.697202,1.591042,65.0,NaN,3,324253,k92r


In [37]:
single_strategy_ranking_df = ddf[ddf["number of new strategies"] == 1].sort_values("Rank").compute()
single_strategy_ranking_df = single_strategy_ranking_df[["Name", "Mean payoff", "Rank", "Winner"]]
single_strategy_ranking_df = single_strategy_ranking_df.rename(columns={"Mean payoff": "Mean Score"})
single_strategy_ranking_df["Rank"] = single_strategy_ranking_df["Rank"].astype(int)
single_strategy_ranking_df = single_strategy_ranking_df.set_index("Name")
single_strategy_ranking_df.index = single_strategy_ranking_df.index.map(add_superscript_to_name)

In [38]:
alternate_winners_proportion_df = pd.DataFrame()
for i in range(1, 4 + 1):
    alternate_winners_proportion_df = alternate_winners_proportion_df.join(
        pd.DataFrame(ddf[(ddf["number of new strategies"] == i)]["Winner"].value_counts().compute()),
        how="outer",
    )
    total = alternate_winners_proportion_df["count"].sum()
    alternate_winners_proportion_df["count"] = alternate_winners_proportion_df["count"] / total
    alternate_winners_proportion_df = alternate_winners_proportion_df.rename(columns={"count": f"{i} New (N = {int(total / i)})"})

In [39]:
ddf

,Name,Mean payoff,Median payoff,Rank,Original Rank,number of new strategies,tournament id,Winner,Winner score,Tournament average score,k92r score
npartitions=1,,,,,,,,,,,
,string,float64,float64,float64,float64,int64,int64,string,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...


In [40]:
alternate_winners_proportion_df.index = alternate_winners_proportion_df.index.map(add_superscript_to_name)

In [41]:
alternate_winners_proportion_df.tail(8).fillna(0).sum(axis=0)

1 New (N = 209)         1.000000
2 New (N = 21736)       0.998942
3 New (N = 1499784)     0.996743
4 New (N = 77238876)    0.982354
dtype: float64

In [42]:
table = alternate_winners_proportion_df.fillna(0).copy()
table = table[table.index.isin(characteristics.keys())]

table.loc["Sum"] = table.sum(axis=0, numeric_only=True)
table = table.round(5)

table.index = [add_superscript_to_name(i) if i != "Sum" else i for i in table.index]

with open("../paper/assets/extra_strategies_summary.tex", "w") as f:
    f.write(table.to_latex(float_format="%.5f"))

table

,1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
k32r,0.00000,0.00000,0.00001,0.00079
k41r,0.00000,0.00000,0.00001,0.00000
k42r,0.14833,0.26941,0.36640,0.21723
k44r,0.00000,0.00023,0.00057,0.00461
k49r,0.00000,0.00014,0.00035,0.00023
k60r,0.00478,0.01118,0.01882,0.00451
k75r,0.00000,0.00051,0.00245,0.00086
k92r,0.84689,0.71747,0.60814,0.75412
Sum,1.00000,0.99894,0.99674,0.98235


In [30]:
# full_table = alternate_winners_proportion_df.fillna(0).copy()

# # values = [no_addition.get(i, 0) for i in scores_index]
# #full_table.insert(loc=0, column="0 New (N = 1)", value=values)

# full_table.loc["Sum"] = table.sum(axis=0, numeric_only=True)
# full_table = table.round(3)

# full_table.index = [add_superscript_to_name(i) if i != "Sum" else i for i in table.index]

# with open("../paper/assets/extra_strategies_summary_full_table.tex", "w") as f:
#     f.write(full_table.to_latex(float_format="%.3f"))

In [31]:
# full_table

### Intervals

In [46]:

def binomial_ci_from_proportion(p, total, z=1.96):
    se = np.sqrt(p * (1 - p) / total)
    return max(0, p - z * se), min(1, p + z * se)

proportion_tables = []
ci_tables = []

for i in range(1, 5):

    counts = (
        ddf[ddf["number of new strategies"] == i]["Winner"]
        .value_counts()
        .compute()
    )

    total_rows = counts.sum()
    total_tournaments = comb(209, i)

    proportions = counts / total_rows

    proportion_tables.append(
        proportions.rename(f"{i} New (N = {total_tournaments})")
    )

    ci_tables.append(
        proportions.apply(
            lambda p: binomial_ci_from_proportion(p, total_tournaments)
        ).rename(f"{i} New (N = {total_tournaments})")
    )

alternate_winners_proportion_df = pd.concat(
    proportion_tables,
    axis=1
).fillna(0)

alternate_winners_ci_df = pd.concat(
    ci_tables,
    axis=1
)

alternate_winners_proportion_df.index = (
    alternate_winners_proportion_df.index.map(add_superscript_to_name)
)

alternate_winners_ci_df.index = (
    alternate_winners_ci_df.index.map(add_superscript_to_name)
)

table = alternate_winners_proportion_df.copy()
table = table[table.index.isin(characteristics.keys())]

table.loc["Sum"] = table.sum(axis=0, numeric_only=True)
table = table.round(5)

table.index = [
    add_superscript_to_name(i) if i != "Sum" else i
    for i in table.index
]

ci_table = alternate_winners_ci_df.copy()
ci_table = ci_table[ci_table.index.isin(characteristics.keys())]

ci_table = ci_table.applymap(
    lambda interval: (
        f"[{interval[0]:.5f}, {interval[1]:.5f}]"
        if isinstance(interval, tuple)
        else ""
    )
)

ci_table.index = [
    add_superscript_to_name(i)
    for i in ci_table.index
]

/var/folders/95/bfy4zf7n6p5248zz4wn2txfm0000gn/T/ipykernel_33149/292689796.py:67: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ci_table = ci_table.applymap(


In [47]:
ci_table

,1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
k42r,"[0.10014, 0.19651]","[0.26352, 0.27531]","[0.36563, 0.36717]","[0.21714, 0.21732]"
k60r,"[0.00000, 0.01414]","[0.00978, 0.01258]","[0.01860, 0.01904]","[0.00450, 0.00453]"
k92r,"[0.79807, 0.89571]","[0.71149, 0.72346]","[0.60736, 0.60892]","[0.75402, 0.75421]"
k49r,,"[0.00000, 0.00029]","[0.00032, 0.00038]","[0.00023, 0.00023]"
k44r,,"[0.00003, 0.00043]","[0.00053, 0.00061]","[0.00459, 0.00462]"
k75r,,"[0.00021, 0.00081]","[0.00237, 0.00253]","[0.00086, 0.00087]"
k32r,,,"[0.00000, 0.00001]","[0.00078, 0.00080]"
k41r,,,"[0.00001, 0.00002]","[0.00000, 0.00000]"


In [51]:
table = table.reindex(

    ['k32r', 'k41r', 'k42r', 'k44r', 'k49r', 'k60r', 'k75r', 'k92r', 'Sum']

)

In [52]:
table

,1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
k32r,0.00000,0.00000,0.00001,0.00079
k41r,0.00000,0.00000,0.00001,0.00000
k42r,0.14833,0.26941,0.36640,0.21723
k44r,0.00000,0.00023,0.00057,0.00461
k49r,0.00000,0.00014,0.00035,0.00023
k60r,0.00478,0.01118,0.01882,0.00451
k75r,0.00000,0.00051,0.00245,0.00086
k92r,0.84689,0.71747,0.60814,0.75412
Sum,1.00000,0.99894,0.99674,0.98235


In [55]:
ci_table = ci_table.reindex(

    ['k32r', 'k41r', 'k42r', 'k44r', 'k49r', 'k60r', 'k75r', 'k92r']

)

In [56]:
ci_table

,1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
k32r,,,"[0.00000, 0.00001]","[0.00078, 0.00080]"
k41r,,,"[0.00001, 0.00002]","[0.00000, 0.00000]"
k42r,"[0.10014, 0.19651]","[0.26352, 0.27531]","[0.36563, 0.36717]","[0.21714, 0.21732]"
k44r,,"[0.00003, 0.00043]","[0.00053, 0.00061]","[0.00459, 0.00462]"
k49r,,"[0.00000, 0.00029]","[0.00032, 0.00038]","[0.00023, 0.00023]"
k60r,"[0.00000, 0.01414]","[0.00978, 0.01258]","[0.01860, 0.01904]","[0.00450, 0.00453]"
k75r,,"[0.00021, 0.00081]","[0.00237, 0.00253]","[0.00086, 0.00087]"
k92r,"[0.79807, 0.89571]","[0.71149, 0.72346]","[0.60736, 0.60892]","[0.75402, 0.75421]"


In [61]:
with open("../paper/assets/extra_strategies_ci.tex", "w") as f:
    f.write(ci_table.replace("", "-").to_latex(float_format="%.5f"))